# Texas Monthly Electricity Consumption — Regression Project

## Problem Definition
This project estimates monthly electricity consumption in Texas using the average temperature and calendar information for that same month; it is not a forecast made before the weather is known.

The current model uses the observed monthly average temperature. Therefore, this is an estimate when the month's weather is available, not a forecast made before that month begins.

## Target
`consumption` — monthly electricity consumption, in million kilowatt-hours as described by the source data.

## Evaluation Strategy
Because the observations are monthly and time-ordered, the data will be split chronologically:

- **Training set:** used to fit the model.
- **Validation set:** used to compare models and tune hyperparameters.
- **Test set:** the latest observations, used once for the final evaluation after selecting the model.

A random train/test split is not used for final model selection or reporting.


## Load Data and Feature Engineering


In [1]:
import pandas as pd

# تحميل الـ dataset النظيف
df = pd.read_csv('../data/processed/texas_monthly_dataset.csv', parse_dates=['date'])

# إضافة مربع الحرارة - يساعد الموديل يمسك شكل الـ U اللي اكتشفناه في EDA
df['avg_temp_f_sq'] = df['avg_temp_f'] ** 2

# استخراج السنة والشهر من عمود date
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

print(df.head())
print(df.shape)

   consumption       date  avg_temp_f  temp_departure  avg_temp_f_sq  year  \
0        27584 2001-01-01        43.9            -3.1        1927.21  2001   
1        24171 2001-02-01        51.9             1.1        2693.61  2001   
2        22424 2001-03-01        53.3            -6.1        2840.89  2001   
3        22635 2001-04-01        67.8             1.5        4596.84  2001   
4        24516 2001-05-01        74.1             0.2        5490.81  2001   

   month  
0      1  
1      2  
2      3  
3      4  
4      5  
(306, 7)


In [2]:
# تحديد الـ Features (X) - كل الأعمدة اللي هتساعد الموديل يتوقع
X = df[['avg_temp_f', 'avg_temp_f_sq', 'year', 'month']]

# تحديد الـ Target (y) - العمود اللي عايزين نتوقعه
y = df['consumption']

print(X.head())
print(y.head())


   avg_temp_f  avg_temp_f_sq  year  month
0        43.9        1927.21  2001      1
1        51.9        2693.61  2001      2
2        53.3        2840.89  2001      3
3        67.8        4596.84  2001      4
4        74.1        5490.81  2001      5
0    27584
1    24171
2    22424
3    22635
4    24516
Name: consumption, dtype: int64


## Chronological Train / Validation / Test Split


In [3]:
# ترتيب البيانات زمنيًا
df = df.sort_values("date").reset_index(drop=True)

features = ["avg_temp_f", "avg_temp_f_sq", "year", "month"]
target = "consumption"

# تقسيم زمني: 60% تدريب، 20% تحقق، وآخر 20% اختبار
n = len(df)
train_end = int(n * 0.60)
val_end = int(n * 0.80)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

X_test = test_df[features]
y_test = test_df[target]

print("Train:", train_df["date"].min(), "→", train_df["date"].max(), "|", len(train_df))
print("Validation:", val_df["date"].min(), "→", val_df["date"].max(), "|", len(val_df))
print("Test:", test_df["date"].min(), "→", test_df["date"].max(), "|", len(test_df))


Train: 2001-01-01 00:00:00 → 2016-03-01 00:00:00 | 183
Validation: 2016-04-01 00:00:00 → 2021-04-01 00:00:00 | 61
Test: 2021-05-01 00:00:00 → 2026-06-01 00:00:00 | 62


## Baseline and Model Comparison


In [4]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# Baseline: يتوقع دائمًا متوسط الاستهلاك في بيانات التدريب
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_val_pred = baseline.predict(X_val)

# Linear Regression: يتعلم من بيانات التدريب فقط
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_val_pred = linear_model.predict(X_val)

# مقارنة النموذجين على Validation فقط
validation_results = []

for name, predictions in [
    ("Training-mean baseline", baseline_val_pred),
    ("Linear Regression", linear_val_pred),
]:
    validation_results.append({
        "Model": name,
        "Validation R2": r2_score(y_val, predictions),
        "Validation MAE": mean_absolute_error(y_val, predictions),
        "Validation RMSE": np.sqrt(mean_squared_error(y_val, predictions)),
    })

validation_results = pd.DataFrame(validation_results)
display(validation_results)


,Model,Validation R2,Validation MAE,Validation RMSE
0,Training-mean baseline,-1.27835,5691.849144,7399.747810
1,Linear Regression,0.75756,1841.740691,2413.842931


## Time-Series Cross-Validation and Hyperparameter Tuning


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

# تحقق زمني داخل بيانات التدريب فقط
time_cv = TimeSeriesSplit(n_splits=5)

# تجهيز Pipeline لكل نموذج: توحيد الخصائص ثم النموذج
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

lasso_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(max_iter=20000))
])

# تجربة قيم مختلفة للـ regularization باستخدام CV زمني على التدريب
alpha_values = [0.01, 0.1, 1, 10, 100, 1000]

ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid={"model__alpha": alpha_values},
    scoring="neg_root_mean_squared_error",
    cv=time_cv,
    n_jobs=-1
)

lasso_search = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid={"model__alpha": alpha_values},
    scoring="neg_root_mean_squared_error",
    cv=time_cv,
    n_jobs=-1
)

# التدريب والضبط يستخدمان Training فقط
ridge_search.fit(X_train, y_train)
lasso_search.fit(X_train, y_train)

# مقارنة النماذج على Validation فقط
models_to_compare = {
    "Training-mean baseline": baseline,
    "Linear Regression": linear_model,
    "Tuned Ridge": ridge_search.best_estimator_,
    "Tuned Lasso": lasso_search.best_estimator_,
}

comparison_rows = []

for name, fitted_model in models_to_compare.items():
    val_predictions = fitted_model.predict(X_val)

    comparison_rows.append({
        "Model": name,
        "Validation R2": r2_score(y_val, val_predictions),
        "Validation MAE": mean_absolute_error(y_val, val_predictions),
        "Validation RMSE": np.sqrt(
            mean_squared_error(y_val, val_predictions)
        ),
    })

validation_comparison = pd.DataFrame(comparison_rows)
display(validation_comparison)

print("Best Ridge parameters:", ridge_search.best_params_)
print("Best Lasso parameters:", lasso_search.best_params_)


,Model,Validation R2,Validation MAE,Validation RMSE
0,Training-mean baseline,-1.278350,5691.849144,7399.747810
1,Linear Regression,0.757560,1841.740691,2413.842931
2,Tuned Ridge,0.757148,1847.009960,2415.889992
3,Tuned Lasso,0.757555,1841.785728,2413.867093


Best Ridge parameters: {'model__alpha': 0.01}
Best Lasso parameters: {'model__alpha': 0.01}


In [6]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# التحقق الزمني يستخدم Training فقط
time_cv = TimeSeriesSplit(n_splits=5)

# Random Forest: تجربة عدد محدود من الإعدادات
rf_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=1),
    param_grid={
        "n_estimators": [100, 200],
        "max_depth": [3, 6, None],
        "min_samples_leaf": [2, 5],
    },
    scoring="neg_root_mean_squared_error",
    cv=time_cv,
    n_jobs=-1,
)

# Gradient Boosting: تجربة عدد محدود من الإعدادات
gb_search = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid={
        "n_estimators": [100, 200],
        "learning_rate": [0.03, 0.1],
        "max_depth": [1, 2],
    },
    scoring="neg_root_mean_squared_error",
    cv=time_cv,
    n_jobs=-1,
)

# التدريب والضبط على Training فقط
rf_search.fit(X_train, y_train)
gb_search.fit(X_train, y_train)

print("Best Random Forest parameters:", rf_search.best_params_)
print("Best Gradient Boosting parameters:", gb_search.best_params_)

# إضافة النموذجين للمقارنة على Validation
models_to_compare.update({
    "Tuned Random Forest": rf_search.best_estimator_,
    "Tuned Gradient Boosting": gb_search.best_estimator_,
})

comparison_rows = []

for name, fitted_model in models_to_compare.items():
    val_predictions = fitted_model.predict(X_val)

    comparison_rows.append({
        "Model": name,
        "Validation R2": r2_score(y_val, val_predictions),
        "Validation MAE": mean_absolute_error(y_val, val_predictions),
        "Validation RMSE": np.sqrt(
            mean_squared_error(y_val, val_predictions)
        ),
    })

validation_comparison = pd.DataFrame(comparison_rows)
display(validation_comparison)


Best Random Forest parameters: {'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 100}
Best Gradient Boosting parameters: {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 200}


,Model,Validation R2,Validation MAE,Validation RMSE
0,Training-mean baseline,-1.278350,5691.849144,7399.747810
1,Linear Regression,0.757560,1841.740691,2413.842931
2,Tuned Ridge,0.757148,1847.009960,2415.889992
3,Tuned Lasso,0.757555,1841.785728,2413.867093
4,Tuned Random Forest,0.497770,2910.349284,3474.226921
5,Tuned Gradient Boosting,0.648085,2502.787564,2908.210601


## MLflow Experiment Tracking


In [7]:
from pathlib import Path
import os
import mlflow
import mlflow.sklearn


# تحديد مجلد المشروع حتى لو كان الـnotebook مفتوحًا من مجلد notebooks
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

# حفظ سجل MLflow داخل المشروع
mlruns_dir = project_root / "experiments" / "mlruns"
mlruns_dir.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(mlruns_dir.resolve().as_uri())
mlflow.set_experiment("texas_electricity_consumption")

# توثيق قيم الضبط للنماذج التي استخدمت GridSearchCV
best_params_by_model = {
    "Tuned Ridge": ridge_search.best_params_,
    "Tuned Lasso": lasso_search.best_params_,
    "Tuned Random Forest": rf_search.best_params_,
    "Tuned Gradient Boosting": gb_search.best_params_,
}

for model_name, fitted_model in models_to_compare.items():
    if model_name in {"Tuned Random Forest", "Tuned Gradient Boosting"}:
        continue

    val_predictions = fitted_model.predict(X_val)

    with mlflow.start_run(run_name=model_name):
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("evaluation_split", "chronological_validation")
        mlflow.log_param("feature_columns", ", ".join(features))
        mlflow.log_param("train_rows", len(X_train))
        mlflow.log_param("validation_rows", len(X_val))

        if model_name in best_params_by_model:
            for param_name, param_value in best_params_by_model[model_name].items():
                mlflow.log_param(param_name, str(param_value))

        mlflow.log_metric("validation_r2", r2_score(y_val, val_predictions))
        mlflow.log_metric("validation_mae", mean_absolute_error(y_val, val_predictions))
        mlflow.log_metric(
            "validation_rmse",
            np.sqrt(mean_squared_error(y_val, val_predictions))
        )

        mlflow.sklearn.log_model(
            sk_model=fitted_model,
            artifact_path="model"
        )

        print(f"Logged: {model_name} | Run ID: {mlflow.active_run().info.run_id}")

print("MLflow tracking directory:", mlruns_dir)


2026/10/07 21:14:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/07 21:14:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/07 21:14:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logged: Training-mean baseline | Run ID: 8470c3e177444593a139a7eebcdee4e8


2026/10/07 21:15:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/07 21:15:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logged: Linear Regression | Run ID: 7d752cb45e224e5d8811c59086861bc5


2026/10/07 21:15:08 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/07 21:15:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logged: Tuned Ridge | Run ID: e8cddc4e04ba48238c19f859840ae0e0


2026/10/07 21:15:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Logged: Tuned Lasso | Run ID: 3397fcce9da544b18fd7d5231fb419c8
MLflow tracking directory: C:\Users\cavo center\texas-elec\experiments\mlruns


In [8]:
tree_models = {
    "Tuned Random Forest": rf_search.best_estimator_,
    "Tuned Gradient Boosting": gb_search.best_estimator_,
}

tree_searches = {
    "Tuned Random Forest": rf_search,
    "Tuned Gradient Boosting": gb_search,
}

for model_name, fitted_model in tree_models.items():
    val_predictions = fitted_model.predict(X_val)

    with mlflow.start_run(run_name=model_name):
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("evaluation_split", "chronological_validation")
        mlflow.log_param("feature_columns", ", ".join(features))
        mlflow.log_param("train_rows", len(X_train))
        mlflow.log_param("validation_rows", len(X_val))

        for param_name, param_value in tree_searches[model_name].best_params_.items():
            mlflow.log_param(param_name, str(param_value))

        mlflow.log_metric("validation_r2", r2_score(y_val, val_predictions))
        mlflow.log_metric("validation_mae", mean_absolute_error(y_val, val_predictions))
        mlflow.log_metric(
            "validation_rmse",
            np.sqrt(mean_squared_error(y_val, val_predictions))
        )

        mlflow.sklearn.log_model(
            sk_model=fitted_model,
            name="model",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )

        print(f"Logged: {model_name} | Run ID: {mlflow.active_run().info.run_id}")


2026/10/07 21:15:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Logged: Tuned Random Forest | Run ID: dc6b8b552d38425ea01a69c54194990e


2026/10/07 21:15:27 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Logged: Tuned Gradient Boosting | Run ID: c25ac3b753474756a4cb5bae8cc7c776


## Final Test Evaluation


In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import mlflow
import mlflow.sklearn

# النموذج المختار حسب أداء Validation
best_model_name = "Linear Regression"

# بعد الاختيار: نعيد التدريب على Train + Validation
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])

final_model = LinearRegression()
final_model.fit(X_train_val, y_train_val)

# التقييم النهائي على Test
test_predictions = final_model.predict(X_test)

final_test_metrics = {
    "test_r2": r2_score(y_test, test_predictions),
    "test_mae": mean_absolute_error(y_test, test_predictions),
    "test_rmse": np.sqrt(mean_squared_error(y_test, test_predictions)),
}

print("Final model:", best_model_name)
for metric_name, value in final_test_metrics.items():
    print(f"{metric_name}: {value:.4f}")

# تسجيل التقييم النهائي في MLflow
with mlflow.start_run(run_name="FINAL_TEST_Linear_Regression"):
    mlflow.log_param("selected_model", best_model_name)
    mlflow.log_param("selection_basis", "chronological_validation")
    mlflow.log_param("final_training_rows", len(X_train_val))
    mlflow.log_param("test_rows", len(X_test))

    mlflow.log_metrics(final_test_metrics)

    mlflow.sklearn.log_model(
        sk_model=final_model,
        name="final_model"
    )

print("Final test evaluation logged to MLflow.")


Final model: Linear Regression
test_r2: 0.4168
test_mae: 3707.6046
test_rmse: 4250.8981


2026/10/07 21:15:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Final test evaluation logged to MLflow.


## Save Final Pipeline and Model Card


In [10]:
from pathlib import Path
from datetime import datetime
import json
import joblib

# تحديد مجلد المشروع
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

# إنشاء مجلد لحفظ النموذج
model_dir = project_root / "models"
model_dir.mkdir(parents=True, exist_ok=True)

# حفظ النموذج النهائي
model_path = model_dir / "texas_electricity_linear_regression.joblib"
joblib.dump(final_model, model_path)

# إعداد بطاقة تعريف النموذج
model_card = {
    "model_name": "Linear Regression",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset": "Texas monthly electricity consumption",
    "target": "consumption",
    "target_unit": "million kilowatt-hours (as described by source data)",
    "prediction_scope": (
        "Estimate monthly consumption when that month's observed average "
        "temperature is available"
    ),
    "features": features,
    "feature_engineering": {
        "avg_temp_f_sq": "avg_temp_f ** 2",
        "year": "year extracted from date",
        "month": "month number extracted from date",
    },
    "selection": {
        "selected_using": "chronological validation set",
        "reason": "Linear Regression had the lowest validation RMSE among compared models",
        "validation_r2": float(
            validation_comparison.loc[
                validation_comparison["Model"] == "Linear Regression",
                "Validation R2"
            ].iloc[0]
        ),
        "validation_mae": float(
            validation_comparison.loc[
                validation_comparison["Model"] == "Linear Regression",
                "Validation MAE"
            ].iloc[0]
        ),
        "validation_rmse": float(
            validation_comparison.loc[
                validation_comparison["Model"] == "Linear Regression",
                "Validation RMSE"
            ].iloc[0]
        ),
    },
    "final_test_metrics": {
        name: float(value) for name, value in final_test_metrics.items()
    },
    "data_split": {
        "train_period": (
            f"{train_df['date'].min().date()} to "
            f"{train_df['date'].max().date()}"
        ),
        "validation_period": (
            f"{val_df['date'].min().date()} to "
            f"{val_df['date'].max().date()}"
        ),
        "test_period": (
            f"{test_df['date'].min().date()} to "
            f"{test_df['date'].max().date()}"
        ),
        "final_training_rows": int(len(X_train_val)),
        "validation_rows": int(len(X_val)),
        "test_rows": int(len(X_test)),
    },
    "limitations": [
        "The model uses the observed average temperature for the same month.",
        "This is not a forecast issued before that month's weather is known.",
        (
            "The test period was inspected in earlier exploratory modeling; "
            "the final test metrics should therefore be interpreted transparently."
        ),
        "Performance may change over time as electricity demand patterns evolve.",
    ],
}

card_path = model_dir / "model_card.json"
with open(card_path, "w", encoding="utf-8") as f:
    json.dump(model_card, f, indent=2, ensure_ascii=False)

print("Saved model:", model_path)
print("Saved model card:", card_path)


Saved model: C:\Users\cavo center\texas-elec\models\texas_electricity_linear_regression.joblib
Saved model card: C:\Users\cavo center\texas-elec\models\model_card.json


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from pathlib import Path
import numpy as np
import json
import joblib
import mlflow
import mlflow.sklearn

# ندرّب النموذج النهائي على Train + Validation بعد اختيار النموذج
train_val_df = pd.concat([train_df, val_df]).sort_values("date")

raw_features = ["avg_temp_f", "year", "month"]

X_train_val_raw = train_val_df[raw_features]
y_train_val_final = train_val_df["consumption"]

X_test_raw = test_df[raw_features]
y_test_final = test_df["consumption"]

# Pipeline: إنشاء مربع الحرارة تلقائيًا ثم تدريب الانحدار الخطي
feature_builder = ColumnTransformer(
    transformers=[
        (
            "temperature",
            PolynomialFeatures(degree=2, include_bias=False),
            ["avg_temp_f"],
        ),
        ("calendar", "passthrough", ["year", "month"]),
    ],
    remainder="drop",
)

final_pipeline = Pipeline([
    ("feature_engineering", feature_builder),
    ("regressor", LinearRegression()),
])

final_pipeline.fit(X_train_val_raw, y_train_val_final)

# تقييم Pipeline على Test
pipeline_test_predictions = final_pipeline.predict(X_test_raw)

pipeline_test_metrics = {
    "test_r2": r2_score(y_test_final, pipeline_test_predictions),
    "test_mae": mean_absolute_error(y_test_final, pipeline_test_predictions),
    "test_rmse": np.sqrt(
        mean_squared_error(y_test_final, pipeline_test_predictions)
    ),
}

print("Pipeline test metrics:")
for metric, value in pipeline_test_metrics.items():
    print(f"{metric}: {value:.4f}")

# حفظ Pipeline كاملة
pipeline_path = model_dir / "texas_electricity_pipeline.joblib"
joblib.dump(final_pipeline, pipeline_path)
print("Saved complete preprocessing + model pipeline:", pipeline_path)

# تحديث بطاقة النموذج لتوضح ملف الـPipeline ومدخلاتها الخام
card_path = model_dir / "model_card.json"

with open(card_path, "r", encoding="utf-8") as f:
    model_card = json.load(f)

model_card["pipeline_file"] = pipeline_path.name
model_card["raw_input_features"] = raw_features
model_card["feature_engineering"] = (
    "PolynomialFeatures(degree=2) for avg_temp_f; "
    "year and month passed through"
)
model_card["pipeline_test_metrics"] = {
    name: float(value) for name, value in pipeline_test_metrics.items()
}

with open(card_path, "w", encoding="utf-8") as f:
    json.dump(model_card, f, indent=2, ensure_ascii=False)

# تسجيل الـPipeline والنتائج في MLflow
with mlflow.start_run(run_name="FINAL_TEST_Linear_Pipeline"):
    mlflow.log_param("selected_model", "Linear Regression Pipeline")
    mlflow.log_param("raw_input_features", ", ".join(raw_features))
    mlflow.log_param("selection_basis", "chronological validation")
    mlflow.log_metrics(pipeline_test_metrics)

    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        name="final_pipeline",
    )

print("Pipeline and final metrics logged to MLflow.")
print("Updated model card:", card_path)


Pipeline test metrics:
test_r2: 0.4168
test_mae: 3707.6046
test_rmse: 4250.8981
Saved complete preprocessing + model pipeline: C:\Users\cavo center\texas-elec\models\texas_electricity_pipeline.joblib


2026/10/07 21:23:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Pipeline and final metrics logged to MLflow.
Updated model card: C:\Users\cavo center\texas-elec\models\model_card.json
